# Mock Exam 2 — Classification → Regression (ภาษาไทย)
## Train.csv / Test.csv • 2 Targets • End-to-end Notebook

**โจทย์จำลอง:** จากข้อมูลลูกค้าให้ทำนาย
1. `purchase_class` — ซื้อสินค้าไหม (0/1) **Classification**
2. `spend_target` — ยอดใช้จ่าย (ตัวเลขต่อเนื่อง) **Regression**

โดยต้องนำผลจาก Classifier มาเป็น Feature สำหรับ Regressor และ **ห้ามใช้คำตอบจริงของ Test** เป็น Feature

> Notebook นี้เป็นโจทย์จำลอง ไม่ใช่ข้อสอบจริงของอาจารย์ และเลขเซลล์อ้างอิงด้านล่างเป็นหัวข้อที่ควรไปเทียบใน Notebook อาจารย์ ไม่ใช่การรับรองว่าโค้ดเหมือนกันทุกบรรทัด


## Slide 1 — แผนการทำข้อสอบ
`train.csv` → แยก X, y_class, y_reg → preprocess → Classifier → out-of-fold probability บน Train → Regressor ที่รับ X + probability → ทำนาย test.csv ทั้ง 2 target → Evaluate / Export

**อ้างอิงสำหรับทบทวน:** `W2_4_FeatureEngineering.ipynb` (encoding/scaling/missing), `W4-1_Logistic Regression.ipynb` (classification), `W5-2_Random_Forest.ipynb` (ensemble), `W3-1b_Linear_PolynomialRegression.ipynb` (regression)

**กติกา:** ข้อนี้สมมติว่าใช้ sklearn ได้


In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score

SEED=42
TRAIN_PATH='Mock_ClassToReg_Train.csv'
TEST_PATH='Mock_ClassToReg_Test.csv'
UPLOAD_FILES=True   # Colab: True ให้ upload CSV สองไฟล์
CLASS_TARGET='purchase_class'
REG_TARGET='spend_target'


## Slide 2 — Import CSV / ดูชนิดข้อมูล
**อ้างอิง:** `W2_4_FeatureEngineering.ipynb` ส่วน DataFrame, missing values

ไฟล์ Mock Test มี Target ให้ด้วยเพื่อ **ตรวจคะแนนฝึก**; ข้อสอบจริงอาจให้ Test ที่ไม่มี Target และต้องส่ง Predictions อย่างเดียว


In [14]:
train=pd.read_csv(TRAIN_PATH)
test=pd.read_csv(TEST_PATH)
print('Train:',train.shape,'Test:',test.shape)
display(train.head())
display(train.dtypes.to_frame('dtype'))
display(train.isna().sum().to_frame('missing'))


Train: (1000, 8) Test: (250, 8)


,age,monthly_income_k,visits,tenure_years,membership,channel,purchase_class,spend_target
0,69,27.77,2,4.22,Gold,Web,0,91.77
1,69,55.62,4,1.19,Silver,Store,1,244.33
2,66,61.21,3,6.09,Basic,App,1,271.78
3,23,59.65,1,2.64,Basic,Web,0,123.34
4,52,75.11,2,7.73,Gold,App,1,305.97


,dtype
age,int64
monthly_income_k,float64
visits,int64
tenure_years,float64
membership,object
channel,object
purchase_class,int64
spend_target,float64


,missing
age,0
monthly_income_k,27
visits,0
tenure_years,0
membership,25
channel,16
purchase_class,0
spend_target,0


## Slide 3 — Define X and Targets; protect Test
- `X` = Features ที่มีอยู่ตอนใช้งานจริง
- `y_class` = Label สำหรับ Classifier
- `y_reg` = จำนวนเงินสำหรับ Regressor
- **อย่าใส่ `purchase_class` จริงเป็น feature ของ Regressor ใน Test**
- หาก Test ไม่มี Labels จะประเมิน Test ไม่ได้ แต่ Export Predictions ได้


In [15]:
assert CLASS_TARGET in train and REG_TARGET in train
train=train.dropna(subset=[CLASS_TARGET,REG_TARGET]).copy()
DROP_COLS=[]  # ใส่ ID หรือ leakage columns เช่น ['customer_id']
feature_cols=[c for c in train.columns if c not in [CLASS_TARGET,REG_TARGET]+DROP_COLS]
X=train[feature_cols]
yc=train[CLASS_TARGET].astype(int)
yr=train[REG_TARGET].astype(float)
X_test=test[feature_cols]
has_test_class=CLASS_TARGET in test.columns and test[CLASS_TARGET].notna().all()
has_test_reg=REG_TARGET in test.columns and test[REG_TARGET].notna().all()
print('Features:',feature_cols)
print('Class distribution:'); display(yc.value_counts(normalize=True))


Features: ['age', 'monthly_income_k', 'visits', 'tenure_years', 'membership', 'channel']
Class distribution:


purchase_class
0    0.566
1    0.434
Name: proportion, dtype: float64

## Slide 4 — Preprocess numeric + categorical
**อ้างอิง:** `W2_4_FeatureEngineering.ipynb` scaling/encoding/missing values

- Numerical: median → StandardScaler
- Categorical: most frequent → OneHotEncoder(ignore unknown)
- Preprocessing อยู่ **ใน Pipeline** เพื่อให้ cross-validation fit ใหม่ในแต่ละ Fold และลด Leakage


In [16]:
numeric_cols=X.select_dtypes(include=np.number).columns.tolist()
categorical_cols=[c for c in feature_cols if c not in numeric_cols]
def make_preprocessor():
    return ColumnTransformer([
        ('num',Pipeline([('impute',SimpleImputer(strategy='median')),
                         ('scale',StandardScaler())]),numeric_cols),
        ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                         ('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical_cols)
    ],sparse_threshold=0)
print('Numeric:',numeric_cols)
print('Categorical:',categorical_cols)


Numeric: ['age', 'monthly_income_k', 'visits', 'tenure_years']
Categorical: ['membership', 'channel']


## Slide 5 — Classification: train probability with Out-of-Fold (OOF)
**อ้างอิง:** `W4-1_Logistic Regression.ipynb` (Logistic Regression), `W5-2_Random_Forest.ipynb` (Random Forest)

**ทำไม OOF?** หากใช้ Classifier ทำนาย Train ชุดเดียวกับที่เรียนมา ผลอาจดีเกินจริง ทำให้ Regressor เรียนรู้จากข้อมูลที่ไม่เหมือนตอนสอบ

`cross_val_predict(..., method='predict_proba')` ให้แต่ละแถวของ Train ถูกทำนายจากโมเดลที่ไม่ได้ฝึกบนแถวนั้น

ใช้ probability ของ class 1 แทน hard label เพื่อรักษาข้อมูลความมั่นใจของโมเดล


In [23]:
from lightgbm import LGBMClassifier, LGBMRegressor

clf=Pipeline([
    ('prep',make_preprocessor()),
    ('model',RandomForestClassifier(
        n_estimators=220,min_samples_leaf=3,
        random_state=SEED,n_jobs=-1))
])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
oof_proba=cross_val_predict(clf,X,yc,cv=cv,method='predict_proba',n_jobs=1)[:,1]
print('OOF Classification Accuracy:',accuracy_score(yc,(oof_proba>=0.5).astype(int)))
print('OOF F1:',f1_score(yc,(oof_proba>=0.5).astype(int)))
print('OOF ROC AUC:',roc_auc_score(yc,oof_proba))
# fit final classifier on ALL training rows, used only to predict Test
clf.fit(X,yc)
test_proba=clf.predict_proba(X_test)[:,1]
test_class=(test_proba>=0.5).astype(int)


OOF Classification Accuracy: 0.668
OOF F1: 0.5990338164251208
OOF ROC AUC: 0.7176157365944212


## Slide 6 — Regression: X + classifier probability
**อ้างอิง:** `W3-1b_Linear_PolynomialRegression.ipynb` (regression); `W5-2_Random_Forest.ipynb` (tree-based models)

Train Regression บน Features ปกติ + `class_probability` จาก OOF

Test Regression บน Features ปกติ + `class_probability` จาก Classifier ที่ fit บน Train ทั้งหมด

**ข้อจำกัด:** Probability ของ OOF และ final Classifier อาจมี distribution ต่างกันเล็กน้อย; เป็นวิธี practical baseline ไม่ใช่การรับประกันว่า score จะสูงขึ้น


In [24]:
X_reg=X.copy()
X_reg['class_probability']=oof_proba
X_test_reg=X_test.copy()
X_test_reg['class_probability']=test_proba

# Regression pipeline, including preprocessing of original features
reg_numeric=numeric_cols+['class_probability']
reg_prep=ColumnTransformer([
    ('num',Pipeline([('impute',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler())]),reg_numeric),
    ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                     ('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical_cols)
],sparse_threshold=0)
reg=Pipeline([
    ('prep',reg_prep),
    ('model',LGBMRegressor(random_state=SEED))
])
reg.fit(X_reg,yr)
test_spend=reg.predict(X_test_reg)
print('First predictions:',test_spend[:5])


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000267 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 841
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 11
[LightGBM] [Info] Start training from score 214.004500
First predictions: [156.81603804 200.66885322 123.25343065  65.37392498  84.93522203]


## Slide 7 — Evaluate (เฉพาะกรณี Test มีเฉลย)
**อ้างอิง:** `W3-1a_Linear-PolynomialRegression_Hours2GPA.ipynb` (R²), `W4-1_Logistic Regression.ipynb` (classification evaluation)

- Classification: Accuracy, F1, ROC-AUC
- Regression: MAE, RMSE, R²
- Test ต้องไม่ใช้ฝึกโมเดลหรือเลือก hyperparameters


In [25]:
if has_test_class:
    actual_c=test[CLASS_TARGET].astype(int)
    print('TEST classification Accuracy:',accuracy_score(actual_c,test_class))
    print('TEST classification F1:',f1_score(actual_c,test_class))
    print('TEST classification ROC AUC:',roc_auc_score(actual_c,test_proba))
else:
    print('Test has no classification labels; skipping classification metrics')
if has_test_reg:
    actual_r=test[REG_TARGET].astype(float)
    print('TEST regression MAE:',mean_absolute_error(actual_r,test_spend))
    print('TEST regression RMSE:',np.sqrt(mean_squared_error(actual_r,test_spend)))
    print('TEST regression R2:',r2_score(actual_r,test_spend))
else:
    print('Test has no regression labels; skipping regression metrics')


TEST classification Accuracy: 0.64
TEST classification F1: 0.5631067961165048
TEST classification ROC AUC: 0.68707616066771
TEST regression MAE: 54.93739645367774
TEST regression RMSE: 66.74184579789443
TEST regression R2: 0.5441134316493912


## Slide 8 — Baseline Comparison: Regression ไม่ใช้ Classification
การเพิ่ม Probability **ไม่ได้ดีขึ้นเสมอไป** เปรียบเทียบกับ Regression ที่ใช้ X อย่างเดียว

เพื่อเลือกโมเดลอย่างเป็นธรรมควรทำ validation ภายใน Train (nested CV หรือ holdout) โดยไม่ใช้ Test ในการตัดสินใจ ตัวอย่างด้านล่างเป็นการตรวจ Test แบบสาธิตเท่านั้น


In [26]:
baseline=Pipeline([
    ('prep',make_preprocessor()),
    ('model',LGBMRegressor(random_state=SEED))
])
baseline.fit(X,yr)
baseline_pred=baseline.predict(X_test)
if has_test_reg:
    display(pd.DataFrame([
        {'model':'X only','RMSE':np.sqrt(mean_squared_error(actual_r,baseline_pred)),
         'MAE':mean_absolute_error(actual_r,baseline_pred),'R2':r2_score(actual_r,baseline_pred)},
        {'model':'X + OOF Class Probability','RMSE':np.sqrt(mean_squared_error(actual_r,test_spend)),
         'MAE':mean_absolute_error(actual_r,test_spend),'R2':r2_score(actual_r,test_spend)}
    ]))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 586
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 10
[LightGBM] [Info] Start training from score 214.004500


,model,RMSE,MAE,R2
0,X only,67.176619,54.113494,0.538155
1,X + OOF Class Probability,66.741846,54.937396,0.544113


## Slide 9 — Export Submission CSV
ถ้าอาจารย์ให้ Test ไม่มี Target ให้ส่งไฟล์ `submission.csv` ที่มีสองคอลัมน์ Prediction

กรณีมี `id` ต้องเก็บลำดับแถวและ ID ตามรูปแบบที่โจทย์กำหนด


In [21]:
submission=pd.DataFrame({
    CLASS_TARGET:test_class,
    REG_TARGET:test_spend
})
submission.to_csv('submission.csv',index=False)
display(submission.head())
print('Saved submission.csv')
# Colab download (uncomment):
# from google.colab import files
# files.download('submission.csv')


,purchase_class,spend_target
0,0,158.983985
1,0,193.036448
2,0,124.885414
3,0,64.436378
4,0,103.313463


Saved submission.csv


## Slide 10 — Cheat Sheet / คำถามอาจารย์
**Q: ทำไมไม่ใช้ `purchase_class` จริงของ Train เป็น Feature โดยตรง?**  
A: เพราะตอน Test ไม่มีเฉลย Classification; การใช้ True Label ฝึก Regression จะทำให้ Train/Test ไม่เหมือนกัน (teacher forcing mismatch) และอาจเกิด leakage

**Q: OOF คืออะไร?**  
A: แต่ละแถวได้ prediction จากโมเดลที่ไม่ได้ฝึกบนแถวนั้น

**Q: ทำไมใช้ probability ไม่ใช้ 0/1?**  
A: Probability บอกระดับความมั่นใจและอาจให้ข้อมูลมากกว่า แต่ต้องวัดผลจริง

**Q: Classification score สูง แต่ Regression score ต่ำได้ไหม?**  
A: ได้ เพราะโจทย์วัดคนละ Target; Feature เดียวกันอาจไม่เพียงพอทำนายจำนวนเงิน

**Q: ถ้า Test มีแต่ X ทำไง?**  
A: Predict class probability → append feature → predict regression → export CSV

**Q: ถ้า Classifier ดีแต่ pipeline แย่?**  
A: เช็ก leakage, calibration, overfit, OOF distribution, preprocessing และ baseline

**ข้อควรระวัง:** หากต้องการ tune pipeline จริง ควรแบ่ง Validation ออกจาก Train ก่อนทำ OOF แล้วประเมิน pipeline ทั้งชุดบน Validation; อย่าใช้ Test เพื่อเลือกโมเดล
